# Custom Dataset Evaluation Tutorial

This notebook demonstrates how to evaluate your model against **custom query-answer datasets**.

**What you'll learn:**
- Loading evaluation datasets from CSV files
- Defining async model callables for evaluation
- Running evaluations with an LLM judge
- Analyzing results by category and difficulty

## Setup

First, let's install dependencies and set up API keys.

In [1]:
%pip install -q tavily-agent-toolkit tavily-python langchain langchain-openai langchain-community pandas


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("TAVILY_API_KEY:\n")

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY:\n")

In [4]:
# Import the evaluation framework
from tavily_agent_toolkit import ModelConfig, ModelObject
from tavily_agent_toolkit.evals import (
    DatasetEvaluator,
    CSVDataset,
    InMemoryDataset,
    DatasetItem,
    evaluate_dataset,
)

# Configure the judge model (used to grade correctness)
judge_config = ModelConfig(
    model=ModelObject(
        model="gpt-4o-mini",
        api_key=os.environ["OPENAI_API_KEY"],
    ),
)

## Creating a Dataset

You can create datasets in two ways:
1. Load from a CSV file
2. Create programmatically in memory

### CSV Format

The CSV format requires:
- `query`: The question to ask your model
- `expected_answer`: The correct answer for evaluation

Optional columns:
- `category`: Group questions by type (e.g., "factual", "reasoning")
- `difficulty`: Difficulty level (e.g., "easy", "medium", "hard")
- `metadata`: JSON string with additional data

In [ ]:
# Create a sample dataset programmatically
# These are all current events that require web search to answer correctly
sample_items = [
    # Technology - Programming Languages & Frameworks
    DatasetItem(
        query="What is the latest stable version of Python?",
        expected_answer="3.14.3",
    ),
    DatasetItem(
        query="What is the current LTS version of Node.js?",
        expected_answer="24",
    ),
    DatasetItem(
        query="What is the codename for Node.js 24?",
        expected_answer="Krypton",
    ),
    # AI News
    DatasetItem(
        query="Which AI model was used to plan the first AI-assisted drive on Mars?",
        expected_answer="Claude",
    ),
    DatasetItem(
        query="Which Chinese AI company released the R1 model in January 2025?",
        expected_answer="DeepSeek",
    ),
    DatasetItem(
        query="What was Anthropic's valuation after its $13 billion Series F round in 2025?",
        expected_answer="$183 billion",
    ),
    # Current Events
    DatasetItem(
        query="Who is the current Prime Minister of Canada?",
        expected_answer="Mark Carney",
    ),
    DatasetItem(
        query="What film won Best Picture at the 2025 Academy Awards?",
        expected_answer="Anora",
    ),
    DatasetItem(
        query="Who won Album of the Year at the 2025 Grammy Awards?",
        expected_answer="Beyoncé",
    ),
    DatasetItem(
        query="Who won Super Bowl LIX in 2025?",
        expected_answer="Philadelphia Eagles",
    ),
]

dataset = InMemoryDataset(sample_items)
print(f"Dataset loaded with {len(dataset)} items")

In [ ]:
# Alternative: Save to CSV and load it back
import pandas as pd

# Create a sample CSV with current tech questions
csv_data = {
    "query": [
        "What is the latest stable version of Python?",
        "What is the current LTS version of Node.js?",
        "What is the latest Claude model from Anthropic?",
    ],
    "expected_answer": [
        "3.14.3",
        "24",
        "Claude Opus 4.5",
    ],
}

df = pd.DataFrame(csv_data)
df.to_csv("sample_dataset.csv", index=False)
print("Saved sample_dataset.csv")
print(df)

In [ ]:
# Load from CSV
csv_dataset = CSVDataset("sample_dataset.csv")
print(f"Loaded {len(csv_dataset)} items from CSV")

for item in csv_dataset:
    print(f"  Q: {item.query}")
    print(f"  A: {item.expected_answer}")
    print(f"  Category: {item.category}")
    print()

## Defining Your Model

Your model must be an async callable with this signature:

```python
async def model(query: str) -> str:
    # Return the answer as a string
    ...
```

Let's create a few example models to compare.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.prompts import ChatPromptTemplate

# Initialize LangChain components
llm = ChatOpenAI(model="gpt-4.1-mini")
tavily_tool = TavilySearchResults(max_results=5, search_depth="advanced")

# Model 1: Simple LLM (no search)
async def llm_only_model(query: str) -> str:
    """Answer using only the LLM's knowledge."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Answer questions concisely and accurately."),
        ("user", "{query}")
    ])
    chain = prompt | llm
    response = await chain.ainvoke({"query": query})
    return response.content


# Model 2: Tavily Search
async def rag_model(query: str) -> str:
    """Answer using Tavily search + LLM."""
    # Search for relevant information
    search_results = await tavily_tool.ainvoke({"query": query})
    
    # Build context from search results
    context = "\n\n".join([
        f"Source: {r['url']}\n{r['content']}"
        for r in search_results
    ])
    
    # Generate answer with context
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Answer the question based on the following context. Be concise.\n\nContext:\n{context}"),
        ("user", "{query}")
    ])
    chain = prompt | llm
    response = await chain.ainvoke({"query": query, "context": context})
    return response.content


# Model 3: Intentionally bad model (for testing)
async def bad_model(query: str) -> str:
    """A bad model that always returns the same answer."""
    return "I don't know."


print("Models defined!")

## Running Evaluation

Now let's evaluate our models against the dataset.

In [ ]:
# Create the evaluator
evaluator = DatasetEvaluator(
    judge_model_config=judge_config,
    parallel=True,  # Run evaluations in parallel
    max_concurrency=5,  # Limit concurrent API calls
)

print("Evaluator ready!")

In [ ]:
# Evaluate the LLM-only model
print("Evaluating LLM-only model...")
llm_result = await evaluator.evaluate(dataset, llm_only_model)

print(f"\nLLM-Only Model Results:")
print(f"  Accuracy: {llm_result.accuracy:.1%}")
print(f"  Correct: {llm_result.correct_count}/{llm_result.total_count}")

In [ ]:
# Evaluate the RAG model
print("Evaluating RAG model...")
rag_result = await evaluator.evaluate(dataset, rag_model)

print(f"\nRAG Model Results:")
print(f"  Accuracy: {rag_result.accuracy:.1%}")
print(f"  Correct: {rag_result.correct_count}/{rag_result.total_count}")

In [ ]:
# Compare the models
print("\n" + "="*50)
print("Model Comparison")
print("="*50)
print(f"LLM-Only: {llm_result.accuracy:.1%} accuracy")
print(f"RAG:      {rag_result.accuracy:.1%} accuracy")
print("="*50)

## Analyzing Results by Category

The evaluator automatically computes breakdowns by category and difficulty.

In [ ]:
# View accuracy by category
print("Accuracy by Category (LLM-Only):")
for cat, breakdown in llm_result.by_category.items():
    print(f"  {cat}: {breakdown.accuracy:.1%} ({breakdown.correct_count}/{breakdown.total_count})")

print("\nAccuracy by Category (RAG):")
for cat, breakdown in rag_result.by_category.items():
    print(f"  {cat}: {breakdown.accuracy:.1%} ({breakdown.correct_count}/{breakdown.total_count})")

In [ ]:
# View accuracy by difficulty
print("Accuracy by Difficulty (RAG):")
for diff, breakdown in rag_result.by_difficulty.items():
    print(f"  {diff}: {breakdown.accuracy:.1%} ({breakdown.correct_count}/{breakdown.total_count})")

## Analyzing Failures

Understanding why your model gets things wrong is key to improvement.

In [ ]:
# Get incorrect answers for analysis
incorrect_items = rag_result.get_incorrect_items()

print(f"Incorrect Answers ({len(incorrect_items)} total):")
print("="*60)

for item in incorrect_items:
    print(f"\nQuery: {item.query}")
    print(f"Expected: {item.expected_answer}")
    print(f"Predicted: {item.predicted_answer}")
    print(f"Category: {item.category}")
    print(f"Reasoning: {item.reasoning}")
    print("-"*40)

In [ ]:
# Export to DataFrame for further analysis
df = rag_result.to_dataframe()

# Filter to incorrect answers
incorrect_df = df[df["grade"] == "INCORRECT"]
print(f"\nIncorrect answers DataFrame:")
print(incorrect_df[["query", "expected_answer", "predicted_answer", "category"]])

## Filtering Datasets

You can filter datasets to evaluate specific subsets.

In [ ]:
# Evaluate only science questions
science_dataset = dataset.filter(category="science")
print(f"Science questions: {len(science_dataset)} items")

science_result = await evaluator.evaluate(science_dataset, rag_model)
print(f"Science accuracy: {science_result.accuracy:.1%}")

In [ ]:
# Evaluate only hard questions
hard_dataset = dataset.filter(difficulty="hard")
print(f"Hard questions: {len(hard_dataset)} items")

if len(hard_dataset) > 0:
    hard_result = await evaluator.evaluate(hard_dataset, rag_model)
    print(f"Hard accuracy: {hard_result.accuracy:.1%}")
else:
    print("No hard questions in the dataset")

## Using the Convenience Function

For quick evaluations, use the `evaluate_dataset` function directly.

In [ ]:
# Quick evaluation with the convenience function
result = await evaluate_dataset(
    dataset=dataset,
    model=llm_only_model,
    judge_model_config=judge_config,
    parallel=True,
)

print(f"Quick eval accuracy: {result.accuracy:.1%}")

## Evaluating Precomputed Predictions

If you've already run your model and have predictions, you can skip the model calls.

In [ ]:
# Precomputed predictions (query, expected, predicted)
precomputed = [
    ("What is 2+2?", "4", "The answer is 4."),
    ("Capital of France?", "Paris", "The capital of France is Paris."),
    ("Who wrote Hamlet?", "Shakespeare", "Charles Dickens"),  # Wrong!
    ("Largest ocean?", "Pacific", "I'm not sure."),  # Wrong!
]

precomputed_result = await evaluator.evaluate_precomputed(
    items=precomputed,
    categories=["math", "geography", "literature", "geography"],
)

print(f"Precomputed accuracy: {precomputed_result.accuracy:.1%}")
print(f"Correct: {precomputed_result.correct_count}/{precomputed_result.total_count}")

## Exporting Results

Results can be exported in various formats for reporting.

In [ ]:
import json

# Export as JSON
result_dict = rag_result.to_dict()
print("Result as JSON:")
print(json.dumps({
    "accuracy": result_dict["accuracy"],
    "correct_count": result_dict["correct_count"],
    "total_count": result_dict["total_count"],
    "by_category": result_dict.get("by_category", {}),
}, indent=2))

In [ ]:
# Export to CSV
df = rag_result.to_dataframe()
df.to_csv("evaluation_results.csv", index=False)
print("Saved evaluation_results.csv")
print(df.head())

## Understanding the Judge

The evaluation uses an LLM judge to determine if answers are correct.

### Grading Criteria

- **CORRECT**: The predicted answer conveys the same essential information as the expected answer. Minor differences in phrasing, formatting, or additional context are acceptable.

- **INCORRECT**: The predicted answer is factually wrong, contradicts the expected answer, misses key information, or fails to answer the question.

### Tips for Good Evaluation

1. **Write clear expected answers** - Be specific about what constitutes a correct answer
2. **Use consistent formatting** - If you expect "Paris", "paris" will still be correct (semantic matching)
3. **Include variations** - Multiple acceptable answers can be captured in the expected answer
4. **Test your dataset** - Run a few manual checks to ensure the judge grades reasonably

In [ ]:
# View judge usage statistics
print("Judge Usage:")
print(f"  LLM calls: {rag_result.usage.judge_llm_calls}")
print(f"  Input tokens: {rag_result.usage.judge_input_tokens}")
print(f"  Output tokens: {rag_result.usage.judge_output_tokens}")
print(f"  Total tokens: {rag_result.usage.judge_total_tokens}")
print(f"  Response time: {rag_result.usage.eval_response_time:.2f}s")

## Cleanup

In [ ]:
# Clean up temporary files
import os

for f in ["sample_dataset.csv", "evaluation_results.csv"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed {f}")

## Summary

In this tutorial, you learned how to:

1. **Create datasets** from CSV files or programmatically
2. **Define model callables** for evaluation
3. **Run evaluations** with parallel processing
4. **Analyze results** by category, difficulty, and individual items
5. **Export results** for reporting

### Next Steps

- Create your own domain-specific dataset
- Compare different model configurations
- Set up automated regression testing
- Combine with other metrics (grounding, relevance) for comprehensive evaluation